In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 159.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 MB 160.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TARGET_COLUMN = "Churn_Value"

# Load feature table from Delta
df = spark.table("telco_churn_features").toPandas()
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Churn rate: {df[TARGET_COLUMN].mean():.3f}")

Loaded: 7043 rows, 31 columns
Churn rate: 0.265


In [0]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} rows")
print(f"Test:  {X_test.shape[0]} rows")
print(f"Churn rate train: {y_train.mean():.3f}")
print(f"Churn rate test:  {y_test.mean():.3f}")

Train: 5634 rows
Test:  1409 rows
Churn rate train: 0.265
Churn rate test:  0.265


In [0]:
mlflow.set_experiment("/churn-retention-system")

with mlflow.start_run(run_name="xgboost_calibrated_isotonic"):

    # --- Base XGBoost pipeline (exact your train.py params) ---
    base_model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    base_pipeline = Pipeline([
        ("model", base_model)
    ])

    base_pipeline.fit(X_train, y_train)
    probs_before = base_pipeline.predict_proba(X_test)[:, 1]

    roc_before = roc_auc_score(y_test, probs_before)
    pr_before = average_precision_score(y_test, probs_before)
    brier_before = brier_score_loss(y_test, probs_before)

    # --- Calibration (exact your calibrate.py) ---
    calibrated = CalibratedClassifierCV(
        estimator=base_pipeline,
        method="isotonic",
        cv=5
    )
    calibrated.fit(X_train, y_train)
    probs_after = calibrated.predict_proba(X_test)[:, 1]

    roc_after = roc_auc_score(y_test, probs_after)
    pr_after = average_precision_score(y_test, probs_after)
    brier_after = brier_score_loss(y_test, probs_after)

    # --- Log params ---
    mlflow.log_param("model", "xgboost")
    mlflow.log_param("calibration", "isotonic")
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    # --- Log metrics ---
    mlflow.log_metric("roc_before", roc_before)
    mlflow.log_metric("pr_before", pr_before)
    mlflow.log_metric("brier_before", brier_before)
    mlflow.log_metric("roc_after", roc_after)
    mlflow.log_metric("pr_after", pr_after)
    mlflow.log_metric("brier_after", brier_after)
    mlflow.log_metric("brier_improvement", brier_before - brier_after)

    # --- Ranking metrics (exact your evaluate.py) ---
    def recall_at_k(y_true, probs, k=0.1):
        d = pd.DataFrame({"y": y_true.values, "p": probs})
        d = d.sort_values("p", ascending=False)
        cut = max(1, int(len(d) * k))
        return d.head(cut)["y"].sum() / d["y"].sum()

    def precision_at_k(y_true, probs, k=0.1):
        d = pd.DataFrame({"y": y_true.values, "p": probs})
        d = d.sort_values("p", ascending=False)
        cut = max(1, int(len(d) * k))
        return d.head(cut)["y"].mean()

    def lift_at_k(y_true, probs, k=0.1):
        d = pd.DataFrame({"y": y_true.values, "p": probs})
        d = d.sort_values("p", ascending=False)
        cut = max(1, int(len(d) * k))
        return d.head(cut)["y"].mean() / d["y"].mean()

    mlflow.log_metric("precision_at_5",  precision_at_k(y_test, probs_after, 0.05))
    mlflow.log_metric("precision_at_10", precision_at_k(y_test, probs_after, 0.10))
    mlflow.log_metric("recall_at_5",     recall_at_k(y_test, probs_after, 0.05))
    mlflow.log_metric("recall_at_10",    recall_at_k(y_test, probs_after, 0.10))
    mlflow.log_metric("lift_at_10",      lift_at_k(y_test, probs_after, 0.10))

    # --- Calibration curve artifact ---
    prob_true, prob_pred = calibration_curve(y_test, probs_after, n_bins=10)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(prob_pred, prob_true, marker="o", label="Calibrated XGBoost")
    ax.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("True probability")
    ax.set_title("Calibration Curve — Isotonic")
    ax.legend()
    fig.savefig("/tmp/calibration_curve.png")
    mlflow.log_artifact("/tmp/calibration_curve.png")
    plt.close()

    # --- Log model ---
    mlflow.sklearn.log_model(calibrated, artifact_path="calibrated_model")

    print(f"\nBefore — ROC: {roc_before:.4f} | PR: {pr_before:.4f} | Brier: {brier_before:.4f}")
    print(f"After  — ROC: {roc_after:.4f} | PR: {pr_after:.4f} | Brier: {brier_after:.4f}")
    print(f"Brier improvement: {brier_before - brier_after:.4f}")

    run_id = mlflow.active_run().info.run_id
    print(f"\nRun ID: {run_id}")

2026/06/06 20:19:17 INFO mlflow.tracking.fluent: Experiment with name '/churn-retention-system' does not exist. Creating a new experiment.
2026/06/06 20:19:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-c6639aca-ea4a.cloud.databricks.com/ml/experiments/1686926800575156/models/m-6f7ff974e07b49f8bef37829d4bb75d6?o=7474650010256733
2026/06/06 20:19:32 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



Before — ROC: 0.8744 | PR: 0.6862 | Brier: 0.1253
After  — ROC: 0.8727 | PR: 0.6800 | Brier: 0.1268
Brier improvement: -0.0015

Run ID: f8a3ddefd04042169bb983288a58781f


In [0]:
model_uri = f"runs:/{run_id}/calibrated_model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name="churn-calibrated-model"
)

print(f"Model registered: {registered.name}")
print(f"Version: {registered.version}")

Successfully registered model 'workspace.default.churn-calibrated-model'.
2026/06/06 20:39:03 WARNING mlflow.tracking._model_registry.fluent: Run with id f8a3ddefd04042169bb983288a58781f has no artifacts at artifact path 'calibrated_model', registering model based on models:/m-6f7ff974e07b49f8bef37829d4bb75d6 instead


---------------------------------------------------------------------------
MlflowException                           Traceback (most recent call last)
File <command-6193038603886807>, line 3
      1 model_uri = f"runs:/{run_id}/calibrated_model"
----> 3 registered = mlflow.register_model(
      4     model_uri=model_uri,
      5     name="churn-calibrated-model"
      6 )
      8 print(f"Model registered: {registered.name}")
      9 print(f"Version: {registered.version}")

File /databricks/python/lib/python3.12/site-packages/mlflow/tracking/_model_registry/fluent.py:136, in register_model(model_uri, name, await_registration_for, tags, env_pack)
     66 def register_model(
     67     model_uri,
     68     name,
   (...)
     72     env_pack: EnvPackType | EnvPackConfig | None = None,
     73 ) -> ModelVersion:
     74     """Create a new model version in model registry for the model files specified by ``model_uri``.
     75 
     76     Note that this method assumes the model registr

In [0]:
from mlflow.models.signature import infer_signature

# Load the already-trained calibrated model from this session
model_uri_logged = f"runs:/{run_id}/calibrated_model"

# Infer signature from training data
signature = infer_signature(X_train, calibrated.predict_proba(X_train)[:, 1])

# Re-log with signature
with mlflow.start_run(run_id=run_id):
    mlflow.sklearn.log_model(
        calibrated,
        name="calibrated_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )

print("Model re-logged with signature")

# Re-register with signature
registered = mlflow.register_model(
    model_uri=f"runs:/{run_id}/calibrated_model",
    name="churn-calibrated-model"
)

print(f"Registered: {registered.name}")
print(f"Version: {registered.version}")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-c6639aca-ea4a.cloud.databricks.com/ml/experiments/1686926800575156/models/m-22764fac36f6409f88dafccd5fbf0896?o=7474650010256733


Model re-logged with signature


Registered model 'churn-calibrated-model' already exists. Creating a new version of this model...
2026/06/06 20:40:38 WARNING mlflow.tracking._model_registry.fluent: Run with id f8a3ddefd04042169bb983288a58781f has no artifacts at artifact path 'calibrated_model', registering model based on models:/m-22764fac36f6409f88dafccd5fbf0896 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered: workspace.default.churn-calibrated-model
Version: 1


🔗 Created version '1' of model 'workspace.default.churn-calibrated-model': https://dbc-c6639aca-ea4a.cloud.databricks.com/explore/data/models/workspace/default/churn-calibrated-model/version/1?o=7474650010256733
